In [1]:
import os
import sys
import platform
import subprocess
from pathlib import Path

print("=== Environment ===")
print(f"OS: {platform.platform()}")
print(f"Python: {sys.version.splitlines()[0]}")
print(f"Executable: {sys.executable}")
print(f"Working dir: {Path.cwd()}")
print(f"VIRTUAL_ENV: {os.getenv('VIRTUAL_ENV')}")
print(f"CONDA_PREFIX: {os.getenv('CONDA_PREFIX')}")

print("\n=== Installed packages (pip list) ===")
subprocess.run([sys.executable, "-m", "pip", "list"], check=False)

=== Environment ===
OS: Linux-6.12.76-linuxkit-aarch64-with-glibc2.41
Python: 3.13.14 (main, Jun 11 2026, 04:57:14) [GCC 14.2.0]
Executable: /workspaces/dtcllm-introduction-to-rag/onnx/.venv/bin/python
Working dir: /workspaces/dtcllm-introduction-to-rag
VIRTUAL_ENV: /workspaces/dtcllm-introduction-to-rag/onnx/.venv
CONDA_PREFIX: None

=== Installed packages (pip list) ===


/workspaces/dtcllm-introduction-to-rag/onnx/.venv/bin/python: No module named pip


CompletedProcess(args=['/workspaces/dtcllm-introduction-to-rag/onnx/.venv/bin/python', '-m', 'pip', 'list'], returncode=1)

In [8]:
from pathlib import Path
print(Path.cwd())

/workspaces/dtcllm-introduction-to-rag


In [9]:
from pathlib import Path
import os

if Path.cwd().name != "onnx":
    os.chdir("onnx")

print(Path.cwd())

/workspaces/dtcllm-introduction-to-rag/onnx


# Download ONNX model

In [4]:
!cd onnx && source .venv/bin/activate && uv run python download.py

tokenizer.json: 100%|███████████████████████| 712k/712k [00:00<00:00, 32.5MB/s]
  saved models/Xenova/all-MiniLM-L6-v2/tokenizer.json
onnx/model.onnx: 100%|████████████████████| 90.4M/90.4M [00:06<00:00, 14.0MB/s]
  saved models/Xenova/all-MiniLM-L6-v2/model.onnx


In [10]:
from embedder import Embedder

embed = Embedder()

q1 = "Can I still join the course after the start date?"
q2 = "How to install Docker on Windows?"
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

v1 = embed.encode(q1)
v2 = embed.encode(q2)
dv = embed.encode(d)

onnxruntime cpuid_info warning: Unknown CPU vendor. cpuinfo_vendor value: 0


Compute similarities

In [11]:
v1.dot(dv)

np.float64(0.32332383260179925)

In [12]:
v2.dot(dv)

np.float64(0.019730417971579945)

Load the documents

In [14]:
from ingest import load_faq_data

documents = load_faq_data()

Combine question and answer for each document

In [15]:
texts = [doc["question"] + " " + doc["answer"] for doc in documents]

Embed in batches

In [16]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/27 [00:00<?, ?it/s]

Search

In [17]:
query = "Can I still join the course after the start date?"
v_query = embed.encode(query)

scores = X.dot(v_query)
idx = np.argmax(scores)

documents[idx]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}